# Búsqueda de configuración (coordenadas)

Búsqueda por coordenadas con el modelo real (`qwen17b`, T4). Mejora una ranura a la vez,
solo acepta si la precisión sube, y repite desde varios arranques al azar.

**Uso:** GPU T4. Ejecute las celdas en orden. Si Colab se desconecta, vuelva a ejecutar todo:
el caché en Drive y las semillas fijas hacen que lo ya generado no se repita.

## Mapa del notebook

Cada sección lleva un código `B1` a `B7` para citarla desde el informe.

| Sección | Qué contiene |
|---|---|
| B1 | Parámetros y selector de arranque (D17) |
| B2 | Prueba corta del caché |
| B3 | Limpieza del historial |
| B4 | Búsqueda por coordenadas y su condición de parada |
| B5 | Análisis de fallos y decisiones derivadas (D6) |
| B6 | Registro de la búsqueda real, leído del repositorio |
| B7 | Validación final con `oraculo.validar` (D12) |

**Método.** Solo se usó la búsqueda por coordenadas (D3); en esta entrega no se compararon otros métodos. Sí se compararon dos arranques con puntos de partida distintos (B6). La configuración reportada es la **mejor encontrada**, no un óptimo demostrado.


In [ ]:
import os  # módulo estándar para consultar y crear archivos y carpetas

MARCA = "/content/.entorno_listo"  # archivo-testigo: si existe, la instalación ya se hizo en esta sesión de Colab
REPO = "https://github.com/Seloga26/ADA2026_2"  # repositorio del equipo
RAMA = "main"  # rama a usar; poner "feat/coordenadas" mientras el PR no esté integrado

if not os.path.exists(MARCA):  # solo instala la primera vez; después se salta todo el bloque
    # Líneas con "!" = comandos de shell en Colab. {RAMA} y {REPO} insertan variables de Python en el comando.
    !pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate nltk spacy emoji langdetect immutabledict
    !python -m spacy download en_core_web_sm -q
    !git clone -q https://github.com/allenai/open-instruct /content/open-instruct
    !git clone -q -b {RAMA} {REPO} /content/ADA2026_2

    import io  # buffers en memoria, para no guardar el .zip en disco
    import urllib.request  # descarga de archivos por HTTP
    import zipfile  # lectura de .zip

    NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"  # origen de los datos de nltk
    PAQUETES = [  # (carpeta, nombre) de cada paquete que necesitan los verificadores de restricciones
        ("tokenizers", "punkt"),
        ("tokenizers", "punkt_tab"),
        ("taggers", "averaged_perceptron_tagger"),
        ("taggers", "averaged_perceptron_tagger_eng"),
    ]
    for carpeta, nombre in PAQUETES:  # descarga y descomprime cada uno
        with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:  # abre la URL; "with" la cierra al terminar
            zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")  # lee los bytes y extrae donde nltk busca

    open(MARCA, "w").close()  # crea el archivo-testigo vacío: la instalación quedó lista
    print("Entorno instalado. El entorno se reinicia: vuelva a ejecutar todas las celdas.")
    os.kill(os.getpid(), 9)  # mata el proceso a propósito: Colab lo reinicia y carga las librerías recién instaladas

# Si el entorno ya existía (instalado desde otro notebook), deja el repo en la rama correcta y al día.
!cd /content/ADA2026_2 && git fetch -q origin && git checkout -q {RAMA} && git pull -q origin {RAMA}
%cd /content/ADA2026_2
print("Entorno listo")

## B1. Parámetros (D17)

`B = 14 400` generaciones en total. El caché y el historial viven en Drive.
Cada arranque se corre por separado para que una desconexión solo cueste ese arranque.

Un único selector: cambie solo `ARRANQUE` (0 o 1); `MAX_PASADAS = 3` vale para los dos.

- **Arranque 0:** `ARRANQUE = 0`, orden fijo y sin exclusiones (`ORDEN_RANURAS = None`, `OPCIONES_EXCLUIDAS = {}`, `MOTIVO = None`).
- **Arranque 1:** `ARRANQUE = 1`; el `if` de la celda aplica el orden y las exclusiones decididas a partir de las trazas del arranque 0.

Esta celda define **todas** las variables que usan las demás celdas. No borre líneas: cambie solo los valores.

In [ ]:
from google.colab import drive  # utilidad propia de Colab para montar Google Drive

drive.mount("/content/drive")  # pide autorización y expone Drive como carpeta
CARPETA = "/content/drive/MyDrive/oraculo_ada"  # carpeta de trabajo en Drive: sobrevive a desconexiones
os.makedirs(CARPETA, exist_ok=True)  # la crea si falta; exist_ok evita error si ya existe


ARRANQUE = 0  # 0 o 1: cambia el punto de partida, las exclusiones y los nombres de archivo
MAX_PASADAS = 3  # tope de pasadas por arranque
B = 14_400  # presupuesto de generaciones (D17); el "_" solo separa miles para leer mejor

SEMILLA = 1  # semilla de generación del modelo (D2)
SEMILLA_BUSQUEDA = 0  # base de la semilla que elige la config inicial al azar
RUTA_CACHE = f"{CARPETA}/oraculo_cache.json"  # caché del oráculo real: lo ya generado no se repite
RUTA_HISTORIAL = f"{CARPETA}/historial_arranque_{ARRANQUE}.csv"  # un CSV de historial por arranque

if ARRANQUE == 0:  # sin decisiones previas: orden fijo y sin exclusiones
    ORDEN_RANURAS = None
    OPCIONES_EXCLUIDAS = {}
    MOTIVO = None
elif ARRANQUE == 1:  # decisión tomada con las trazas del arranque 0 (D6)
    ORDEN_RANURAS = ["formato", "rol", "estilo", "estrategia", "cierre"]
    OPCIONES_EXCLUIDAS = {
        "rol": {0, 3, 4, 7},  # abren con una línea fija y chocan con first_word
        "formato": {1, 4, 5},  # nunca pasaron de 10 % en el arranque 0
        "estrategia": {3},  # nunca pasó de 10 % en el arranque 0
        "cierre": {0, 1, 2, 4, 5, 6, 7},  # solo la 3 no agrega texto al final de la respuesta
    }
    MOTIVO = ("cierre 3 es el unico que no agrega texto al final; rol 0,3,4,7 fijan la primera linea "
              "y chocan con first_word; formato 1,4,5 y estrategia 3 nunca pasaron de 10%")  # queda en la columna decision
else:
    raise ValueError(f"ARRANQUE debe ser 0 o 1, no {ARRANQUE}")
print("Arranque:", ARRANQUE, "| pasadas:", MAX_PASADAS)
print("Historial:", RUTA_HISTORIAL)  # debe terminar en historial_arranque_<ARRANQUE>.csv
assert RUTA_HISTORIAL.endswith(f"historial_arranque_{ARRANQUE}.csv")  # corta aquí si la ruta no coincide con el arranque

In [ ]:
import json  # para escribir resultados en JSON

from ayudas import cargar_datos, cargar_modelo, dividir  # funciones permitidas del repositorio base
from busqueda_prompt.algoritmos import coordenadas  # nuestro algoritmo
from busqueda_prompt.contratos import OPCIONES_POR_RANURA, RANURAS  # 8 opciones y los nombres de las 5 ranuras
from oraculo import Oraculo  # oráculo real (solo funciona en Colab: necesita torch y GPU)

modelo = cargar_modelo("qwen17b")  # descarga y carga el modelo Qwen3-1.7B en la GPU
busqueda, validacion = dividir(cargar_datos())  # parte datos_visibles.json: 100 de búsqueda, 87 de validación
# cache= guarda las generaciones en Drive; lote=16 genera 16 prompts a la vez; presupuesto= es tope de memoria de GPU, NO es B
oraculo = Oraculo(modelo, busqueda, validacion, cache=RUTA_CACHE, lote=16, presupuesto=32_000_000)
print(len(busqueda), "instancias de búsqueda;", len(validacion), "de validación")  # comprobación rápida de tamaños

## B2. Prueba corta del caché (hacerla ANTES de lanzar las 4 horas)

Comprueba dos cosas: que el archivo del caché en Drive **crece** al evaluar, y que repetir las
mismas evaluaciones con un oráculo nuevo (que lee el caché desde Drive) es **casi instantáneo**.
Son solo 10 generaciones (unos 20 s). Si el tamaño no crece o la repetición tarda lo mismo,
el caché no sirve: avisar antes de lanzar la búsqueda.

In [ ]:
import time  # medir tiempos

MUESTRA = busqueda[:10]  # solo 10 instancias: prueba rápida
config_prueba = {"rol": 7, "estrategia": 6, "formato": 5, "estilo": 2, "cierre": 4, "temperatura": 0.0}  # config rara: es poco probable que ya esté en caché


def tamano_cache():  # bytes del archivo de caché en Drive (0 si aún no existe)
    return os.path.getsize(RUTA_CACHE) if os.path.exists(RUTA_CACHE) else 0


antes = tamano_cache()  # tamaño antes de generar
t0 = time.perf_counter()
oraculo.evaluar(config_prueba, MUESTRA, semilla=SEMILLA)  # genera de verdad (o usa caché si ya estaba)
t_primera = time.perf_counter() - t0
despues = tamano_cache()  # tamaño justo después, SIN cerrar la sesión

# Oráculo nuevo con el mismo archivo: simula la sesión siguiente a una desconexión.
oraculo_nuevo = Oraculo(modelo, busqueda, validacion, cache=RUTA_CACHE, lote=16, presupuesto=32_000_000)
t0 = time.perf_counter()
oraculo_nuevo.evaluar(config_prueba, MUESTRA, semilla=SEMILLA)  # debería salir del caché
t_repetida = time.perf_counter() - t0

print(f"tamaño del caché: {antes} -> {despues} bytes  (crece: {despues > antes})")
print(f"primera evaluación: {t_primera:.1f} s   repetida con oráculo nuevo: {t_repetida:.1f} s")
if antes == despues and t_primera < 1.0:  # no creció y fue instantánea: ya estaba en el caché
    print("NO CONCLUYENTE: la config ya estaba en el caché. Si ya hizo la prueba antes, no hace falta repetirla.")
elif despues > antes and t_repetida < 0.2 * t_primera:  # creció y la repetición es mucho más rápida
    print("CACHÉ OK")
else:
    print("REVISAR: el caché no creció o repetir no fue más rápido. Avise antes de lanzar la búsqueda.")


### B3. Limpieza del historial del arranque 0 (solo si hace falta)

Si un intento fallido escribió filas `decide:` en `historial_arranque_0.csv`, esta celda las quita
(hace antes una copia `.copia`). Si no hay nada que quitar, no cambia nada.

In [ ]:
import csv  # lectura y escritura del CSV
import shutil  # copia de seguridad

ruta0 = f"{CARPETA}/historial_arranque_0.csv"  # historial del arranque 0
if os.path.exists(ruta0):  # solo si el archivo existe
    with open(ruta0, encoding="utf-8", newline="") as f:
        filas0 = list(csv.reader(f))  # todas las filas, encabezado incluido
    sobran = [r for r in filas0 if r and r[-1].startswith("decide:")]  # filas de decisión que no pertenecen al arranque 0
    print(len(filas0), "filas en el historial del arranque 0 (esperadas: 76 = encabezado + 75 pasos)")
    if sobran:
        shutil.copy(ruta0, ruta0 + ".copia")  # copia de seguridad antes de tocar nada
        filas0 = [r for r in filas0 if not (r and r[-1].startswith("decide:"))]  # descarta solo esas filas
        with open(ruta0, "w", encoding="utf-8", newline="") as f:
            csv.writer(f).writerows(filas0)  # reescribe el archivo ya limpio
        print("Se quitaron", len(sobran), "filas decide:; ahora hay", len(filas0), "filas")
    else:
        print("Nada que limpiar")
else:
    print("No existe el historial del arranque 0")

## B4. Búsqueda

Un solo arranque por corrida: `semilla_busqueda` y el índice de arranque fijan el punto de
partida, así que repetir la celda reproduce el mismo recorrido (lo ya generado sale del caché).
Cada fila del historial se guarda en Drive al instante.

### B4. Cuándo para la búsqueda

Cada arranque parte de una configuración al azar (semilla `SEMILLA_BUSQUEDA * 1000 + ARRANQUE`) y recorre las ranuras en `ORDEN_RANURAS`. En cada ranura prueba las demás opciones permitidas con el resto fijo, y **cambia solo si la precisión sube estrictamente**: por eso nunca baja.

La corrida termina por la primera de estas razones:

1. **Convergencia:** una pasada completa no acepta ningún cambio. La configuración es un óptimo local por ranura (ningún cambio de una sola ranura la mejora), no un óptimo global.
2. **Tope de pasadas:** se llega a `MAX_PASADAS`.
3. **Presupuesto:** se agota `B`. El algoritmo no falla: devuelve la mejor configuración vista.

Una pasada cuesta como máximo 35 configuraciones por 100 instancias, es decir 3 500 generaciones; lo ya generado sale del caché y no se cobra. Cada corrida lleva su propio contador de `B`: el gasto del proyecto es la **unión** de configuraciones distintas entre los arranques (D11), que calcula `resultados/busqueda/resumir.py`.

En las corridas reales, el arranque 0 paró por convergencia en la pasada 3 (92 configuraciones, 9 200 generaciones) y el arranque 1, en la pasada 2 (28 configuraciones, 2 800 generaciones). Detalle en `resultados/busqueda/analisis_busqueda.md`.


In [ ]:
# Para reanudar tras una desconexión: borre el CSV de este arranque (los pasos se repiten
# desde el caché sin costo de GPU) y vuelva a ejecutar esta celda.
def mostrar_fila(f):  # imprime una fila del historial en pantalla
    if f["precision"] == "":  # la fila "decide: ..." no tiene evaluación: se muestra solo el texto
        print(f"{f['paso']:4d} {f['decision']}")
        return  # sale sin intentar dar formato de porcentaje a un texto vacío
    print(f"{f['paso']:4d} {f['ranura']:>10} {f['opcion']!s:>2} "  # !s pasa a texto; :>10 alinea a la derecha
          f"{f['precision']:6.1%} mejor {f['mejor_precision']:6.1%} "  # :6.1% muestra la fracción como porcentaje
          f"gen {f['consultas']:6d} {f['decision']}")

historial = []  # aquí el algoritmo deja una fila por evaluación, también visible desde el notebook
resultado = coordenadas.resolver(
    oraculo, RANURAS, (OPCIONES_POR_RANURA,) * len(RANURAS), busqueda,  # (8,)*5 = (8, 8, 8, 8, 8): opciones de cada ranura
    semilla=SEMILLA, presupuesto=B, arranques=1, max_pasadas=MAX_PASADAS,  # un arranque por corrida, tope B
    semilla_busqueda=SEMILLA_BUSQUEDA * 1000 + ARRANQUE,  # semilla distinta por arranque, pero fija y reproducible
    historial=historial, ruta_historial=RUTA_HISTORIAL,  # memoria y CSV en Drive
    orden_ranuras=ORDEN_RANURAS, opciones_excluidas=OPCIONES_EXCLUIDAS, motivo=MOTIVO,  # decisión basada en trazas
    mostrar=mostrar_fila,  # función definida arriba: imprime cada fila, también las de decisión
)
print("\nGanadora del arranque:", json.dumps(resultado.ganadora))  # json.dumps muestra la config como texto JSON
print(f"Precisión: {resultado.precision:.1%}   generaciones nuevas: {resultado.consultas}")  # resumen final

In [ ]:
# Resumen para anotar en la bitácora y comparar con el otro arranque.
resumen = {"arranque": ARRANQUE, "config": resultado.ganadora,  # diccionario con lo esencial del arranque
           "precision": resultado.precision, "consultas": resultado.consultas}
with open(f"{CARPETA}/resumen_arranque_{ARRANQUE}.json", "w", encoding="utf-8") as f:  # "w" crea o reescribe; "with" cierra solo
    json.dump(resumen, f, indent=1)  # escribe el diccionario en el archivo, con sangría de 1 espacio
print(resumen)  # lo muestra también en pantalla

## B5. Análisis de fallos (D6): léelo después del arranque 0

Cuenta en cuántas instancias falló la configuración ganadora, por familia de restricción.
Con esto se decide `ORDEN_RANURAS`, `OPCIONES_EXCLUIDAS` y `MOTIVO` (celda de parámetros) y se
corre el arranque 1. Para cada fallo citado en el informe: qué opción se descartó y qué cambió.

In [ ]:
from collections import Counter  # contador de repeticiones

r = oraculo.evaluar(resultado.ganadora, busqueda, semilla=SEMILLA)  # sale del caché: ya se midió al buscar
familia_de = {x["id"]: x["familia"] for x in busqueda}  # id de instancia -> familia de su restricción
fallos_por_familia = Counter(familia_de[t["id"]] for t in r.trazas)  # cuenta fallos por familia
total_familia = Counter(x["familia"] for x in busqueda)  # cuántas instancias hay de cada familia

print(f"Fallos: {len(r.trazas)} de {len(busqueda)}")
for fam, n in fallos_por_familia.most_common():  # de la familia que más falla a la que menos
    print(f"{fam:55s} {n:3d} fallos de {total_familia[fam]:3d}")

print("\nPrimera restricción violada (campo violo), top 10:")
for v, n in Counter(t["violo"] for t in r.trazas).most_common(10):
    print(f"{n:3d}  {v}")

print("\nEjemplos de salida fallida:")
for t in r.trazas[:3]:  # solo 3 para leer
    print("violó:", t["violo"])
    print(t["salida"][:300])
    print("-" * 60)

In [ ]:
# Verificación de la inferencia: ¿alguna opción de la mejor encontrada inserta una frase fija en la respuesta?                           
# Es solo lectura: evalúa desde el caché (sin GPU) y no cambia ningún archivo del repositorio.
from oraculo import CATALOGO  # se consulta imprimiéndolo; no se abre oraculo.py (AGENTS.md, límites de integridad)

FRASE = "Before each part of the answer"  # inicio de la frase repetida en los ejemplos fallidos

def texto_opcion(ranura, opcion):  # texto de una opción del catálogo, sea cual sea su forma
    try:
        return str(CATALOGO[ranura][opcion])
    except Exception:
        return "(no se pudo leer; imprima CATALOGO y ajuste esta función)"

def verificar(config, etiqueta):
    print("=" * 70)
    print(f"{etiqueta}: {json.dumps(config)}")
    for ranura in RANURAS:  # ¿qué opción activa contiene la frase?
        t = texto_opcion(ranura, config[ranura])
        print(f"  {ranura:>10} {config[ranura]}: {'CONTIENE LA FRASE' if FRASE in t else 'no'} | {t[:90]!r}")
    r = oraculo.evaluar(config, busqueda, semilla=SEMILLA)  # sale del caché
    inicio = sum(FRASE in t["salida"][:300] for t in r.trazas)
    print(f"  fallos: {len(r.trazas)} de {len(busqueda)}; contienen la frase en los primeros 300 caracteres: {inicio} de {len(r.trazas)}")

verificar(resultado.ganadora, f"Mejor encontrada del arranque {ARRANQUE}")
ruta_r0 = f"{CARPETA}/resumen_arranque_0.json"  # mejor encontrada del arranque 0, si existe
if os.path.exists(ruta_r0):
    with open(ruta_r0, encoding="utf-8") as f:
        verificar(json.load(f)["config"], "Mejor encontrada del arranque 0")


### B5. Qué enseñaron las trazas

Las trazas del arranque 0 (campo `violo` y familia de cada instancia) llevaron a recortar el arranque 1:

| Ranura | Opciones excluidas | Fundamento |
|---|---|---|
| `rol` | 0, 3, 4, 7 | Prescriben una primera línea literal (texto leído de `CATALOGO`); se supuso que chocan con `first_word` (hipótesis) |
| `formato` | 1, 4, 5 | Nunca pasaron de 10 % en el arranque 0 |
| `estrategia` | 3 | Nunca pasó de 10 % en las pasadas 1 y 2; en la pasada 3 llegó a 15 %, así que la premisa no se sostuvo del todo |
| `cierre` | todas menos la 3 | La 3 es la única que no agrega texto tras la respuesta (comprobado con `CATALOGO`) |

Con esas exclusiones el arranque 1 buscó en 1 120 de las 32 768 configuraciones. En el contexto final del arranque 0, ninguna opción excluida superó 0,15 frente a 0,20 de la mejor encontrada.

Otros hallazgos, con su estado:

- La frase de `estrategia 7` ("Before each part of the answer…") aparece en los primeros 300 caracteres de 32 de 83 fallos del arranque 1 y de 0 de 80 del arranque 0. El origen está comprobado; que cause los fallos, no.
- Las familias que fallan con las dos mejores (copia, `first_word_sent`, `last_word_sent`) casi no dependen de las opciones probadas. Que el modelo copie el texto de las opciones es una hipótesis, no un resultado.

Detalle, pruebas y límites: `resultados/busqueda/analisis_busqueda.md`, secciones 4 a 6.


## B6. Registro de la búsqueda real (desde el repositorio)

Cada arranque tardó horas y no se repite aquí. Su registro está en `resultados/busqueda/` (historiales en CSV y resúmenes en JSON). La celda siguiente lo lee del repositorio clonado, sin GPU ni modelo, y comprueba que los conteos coinciden con los registrados: 92 configuraciones en el arranque 0, 28 en el 1, 117 en la unión y 11 700 generaciones frente a `B`.

Es el registro de las corridas reales, no una nueva ejecución. La mejor precisión observada no es una garantía de optimalidad global.


In [ ]:
import csv
import json
import subprocess
import sys

import busqueda_prompt
from busqueda_prompt.contratos import RANURAS

RAIZ = os.path.dirname(os.path.dirname(busqueda_prompt.__file__))  # raíz del repositorio clonado
RESULTADOS = os.path.join(RAIZ, "resultados", "busqueda")
H0 = os.path.join(RESULTADOS, "historial_arranque_0.csv")
H1 = os.path.join(RESULTADOS, "historial_arranque_1_limpio.csv")  # solo la última ejecución del arranque 1

# resumir.py recalcula los conteos, las 8 mejores y la precisión por opción; falla si no coinciden con lo registrado
salida = subprocess.run(
    [sys.executable, os.path.join(RESULTADOS, "resumir.py"), H0, H1, "--presupuesto", str(B),
     "--esperado-cfg-0", "92", "--esperado-cfg-1", "28", "--esperado-union", "117",
     "--esperado-generaciones", "11700"],
    capture_output=True, text=True,
)
print(salida.stdout)
assert salida.returncode == 0, salida.stderr

# Todas las configuraciones ensayadas, de mayor a menor precisión (una fila por configuración distinta)
vistas = {}
for nombre, ruta_csv in (("arranque 0", H0), ("arranque 1", H1)):
    with open(ruta_csv, encoding="utf-8", newline="") as f:
        for fila in csv.DictReader(f):
            if fila["config"]:
                cfg = json.loads(fila["config"])
                clave = tuple(cfg[r] for r in RANURAS)
                vistas.setdefault(clave, (float(fila["precision"]), nombre))
print(f"{len(vistas)} configuraciones distintas ensayadas (rol, estrategia, formato, estilo, cierre):")
for clave, (precision, nombre) in sorted(vistas.items(), key=lambda kv: -kv[1][0]):
    print(f"  {precision:5.1%}  {clave}  ({nombre})")


## B7. Validación final (D12): una sola vez

**No ejecutar antes del lunes (D9).** `validar` mide la partición de validación sobre la configuración **ya congelada** (D8), una sola vez: si la configuración se eligiera mirando la validación, dejaría de ser una prueba (D12).

La celda de abajo no hace nada mientras `CONFIRMO_VALIDAR` sea `False`, así que "Ejecutar todo" es seguro. Lee la configuración de `resultados/busqueda/resumen_arranque_0.json` (mejor encontrada del arranque 0), comprueba que coincide con `entrega.json` cuando este exista, y llama `oraculo.validar` con `n=None`, que mide toda la partición. El resultado imprime cuántas instancias midió; en los datos visibles se esperan 87.

**Cómo interpretar el resultado.**

- Mide la precisión sobre instancias que no se usaron para elegir la configuración. No es la evaluación final de la docente, que usa datos de prueba no publicados y puede traer familias de restricciones ausentes de los datos visibles.
- El valor de búsqueda (0,20) es el máximo entre 117 configuraciones medidas sobre las mismas 100 instancias, así que probablemente es optimista. Una validación menor no indica un error: es la medida sin ese sesgo.
- Con 87 instancias el ruido de una medida es de unos 4 puntos (estimación teórica, no medida entre semillas).

**Resultado de la validación:** se completa el lunes con la salida de la celda de abajo.


In [ ]:
# Validación (D12): UNA sola vez, el lunes, sobre la configuración congelada (D8).
# Para ejecutarla, cambie CONFIRMO_VALIDAR a True. Con "Ejecutar todo" no hace nada.
CONFIRMO_VALIDAR = False

import json
import os

import busqueda_prompt

RAIZ = os.path.dirname(os.path.dirname(busqueda_prompt.__file__))  # raíz del repositorio clonado
with open(os.path.join(RAIZ, "resultados", "busqueda", "resumen_arranque_0.json"), encoding="utf-8") as f:
    config_congelada = json.load(f)["config"]  # mejor encontrada del arranque 0 (D8)

# Coherencia con entrega.json: la configuración justificada aquí debe ser la que se entrega
ruta_entrega = os.path.join(RAIZ, "entrega.json")
if os.path.exists(ruta_entrega):
    from busqueda_prompt.entrega import cargar_entrega
    assert cargar_entrega(ruta_entrega)["config"] == config_congelada, "entrega.json no coincide con la configuración congelada"
    print("entrega.json coincide con la configuración congelada")
else:
    print("entrega.json aún no existe (se crea en D9)")

if not CONFIRMO_VALIDAR:
    print("Validación NO ejecutada. Configuración congelada:", json.dumps(config_congelada))
else:
    r_val = oraculo.validar(config_congelada, n=None)  # n=None: toda la partición de validación
    print("Configuración:", json.dumps(config_congelada))
    print(f"Validación: {r_val.precision:.1%} sobre {r_val.n} instancias (esperado: 87)")
